# Build a Neuron vLLM BYOC Container for SageMaker

Build a custom BYOC (Bring Your Own Container) for deploying vision-language models on SageMaker Neuron instances (inf2/trn1) using vLLM with NxD Inference.

This notebook walks through the entire container build process from scratch -- no pre-built artifacts or DLAMI extraction required. Everything is installed from public repositories.

## What Gets Built

- **Base**: Official `pytorch-inference-neuronx` DLC (SDK 2.28, PyTorch 2.9)
- **vLLM**: 0.13.0 (installed via vllm-neuron plugin dependencies)
- **vllm-neuron**: 0.4.1 plugin from [vllm-project/vllm-neuron](https://github.com/vllm-project/vllm-neuron)
- **NxD Inference**: neuronx-distributed-inference (from SDK 2.28 DLC)
- **Serving**: FastAPI server at `/ping` + `/invocations` (SageMaker compatible)

## Prerequisites

- **Docker available** (SageMaker notebook instance, EC2, or local machine)
- **AWS CLI configured** with ECR push permissions
- **~20 GB disk space** for the Docker build

## Why SDK 2.28?

SageMaker only offers inf2 and trn1 Neuron instances (no trn2). SDK 2.29 (NxD Inference 0.9) drops inf2/trn1 support, so SDK 2.28 is the latest compatible version.

## Step 1: Configuration

In [ ]:
import boto3
import os
import json

sts = boto3.client('sts')
ACCOUNT_ID = sts.get_caller_identity()['Account']
REGION = boto3.Session().region_name

# Container configuration
ECR_REPO = "qwen3vl-neuron-byoc"  # Change for your model
ECR_TAG = "v1"

# Base DLC image (SDK 2.28)
BASE_IMAGE = f"763104351884.dkr.ecr.{REGION}.amazonaws.com/pytorch-inference-neuronx:2.9.0-neuronx-py312-sdk2.28.0-ubuntu24.04"

# vllm-neuron plugin version (must match SDK)
VLLM_NEURON_VERSION = "0.4.1"  # For SDK 2.28. Use 0.5.0 for SDK 2.29.

IMAGE_URI = f"{ACCOUNT_ID}.dkr.ecr.{REGION}.amazonaws.com/{ECR_REPO}:{ECR_TAG}"

print(f"Account: {ACCOUNT_ID}")
print(f"Region: {REGION}")
print(f"Base image: {BASE_IMAGE}")
print(f"Target image: {IMAGE_URI}")

## Step 2: Create the Dockerfile

The Dockerfile does 4 things:

1. **Starts from the Neuron DLC** -- has torch-neuronx, neuronx-cc, NxD Inference pre-installed
2. **Installs vllm-neuron 0.4.1** from the official GitHub repo -- this pulls vLLM 0.13.0 and registers the Neuron plugin automatically
3. **Patches a known bug** in the Sampler import path
4. **Adds the FastAPI serving code** (model.py)

### Why the Neuron package backup/restore?

When `pip install vllm-neuron` runs, it installs vLLM 0.13.0 which has a transitive dependency on `torch`. pip may try to upgrade/downgrade the DLC's custom `torch-neuronx` build. We save and restore the Neuron packages to prevent this.

In [ ]:
BUILD_DIR = "/tmp/neuron-byoc-build"
os.makedirs(BUILD_DIR, exist_ok=True)
os.makedirs(f"{BUILD_DIR}/code", exist_ok=True)

dockerfile = f"""# Neuron vLLM BYOC Container for SageMaker
# Built entirely from public repositories -- no DLAMI extraction needed.
#
# Base: PyTorch 2.9 + Neuron SDK 2.28 DLC
# Adds: vllm-neuron 0.4.1 plugin (vLLM 0.13.0 + NxD Inference backend)
#
# Supports: Qwen3-VL, Qwen2-VL, Llama, Pixtral, and other NxDI models
# Target: ml.inf2.8xlarge (2 NeuronCores, tp=2)

FROM {BASE_IMAGE}

# SageMaker environment variables
ENV MODEL_CACHE_DIR=/opt/ml/model
ENV TRANSFORMERS_CACHE=/tmp/transformers_cache
ENV HF_HOME=/tmp/hf_home
ENV NEURON_RT_NUM_CORES=2
ENV VLLM_NEURON_FRAMEWORK=neuronx-distributed-inference
ENV NEURON_COMPILE_CACHE_URL=/var/tmp/neuron-compile-cache

# ============================================================
# Step 1: Install vllm-neuron from GitHub
#
# This installs vLLM 0.13.0 + the Neuron plugin. We protect
# the DLC's Neuron packages (torch-neuronx, neuronx-cc, NxDI)
# from being overwritten by pip's dependency resolution.
# ============================================================
RUN SITE_PKG=$(python3 -c "import site; print(site.getsitepackages()[0])") && \\
    mkdir -p /tmp/neuron-backup && \\
    # Save Neuron packages that pip might overwrite
    for pkg in torch_neuronx neuronx_cc neuronx_distributed neuronx_distributed_inference; do \\
      for d in $(find "$SITE_PKG" -maxdepth 1 -name "${{pkg}}*" -type d); do \\
        cp -r "$d" /tmp/neuron-backup/; \\
      done; \\
    done && \\
    # Install vllm-neuron plugin (pulls vLLM 0.13.0 automatically)
    git clone --branch "{VLLM_NEURON_VERSION}" --depth 1 \\
      https://github.com/vllm-project/vllm-neuron.git /tmp/vllm-neuron && \\
    cd /tmp/vllm-neuron && \\
    pip install --no-cache-dir \\
      --extra-index-url=https://pip.repos.neuron.amazonaws.com \\
      -e . && \\
    # Restore Neuron packages
    for item in /tmp/neuron-backup/*/; do \\
      PKG_BASE=$(basename "$item"); \\
      TARGET="$SITE_PKG/$PKG_BASE"; \\
      rm -rf "$TARGET" 2>/dev/null; \\
      cp -r "$item" "$TARGET"; \\
    done && \\
    rm -rf /tmp/neuron-backup && \\
    # Verify everything works
    python3 -c "import torch_neuronx; print('torch-neuronx OK')" && \\
    python3 -c "import vllm; print(f'vLLM {{vllm.__version__}}')" && \\
    python3 -c "import vllm_neuron; print('vllm_neuron plugin OK')"

# ============================================================
# Step 2: Patch known vllm_neuron Sampler import bug
#
# vllm_neuron 0.4.1 imports the sampler module instead of the
# Sampler class, causing TypeError when on-device sampling is
# disabled. This is fixed in 0.5.0 but we need 0.4.1 for SDK 2.28.
# ============================================================
RUN VLLM_NEURON_PATH=$(python3 -c "import vllm_neuron; import os; print(os.path.dirname(vllm_neuron.__file__))") && \\
    sed -i 's|from vllm.v1.sample import sampler as Sampler|from vllm.v1.sample.sampler import Sampler|' \\
      "$VLLM_NEURON_PATH/worker/neuronx_distributed_model_loader.py" && \\
    echo "Patched Sampler import in vllm_neuron"

# ============================================================
# Step 3: Install additional dependencies
# ============================================================
RUN pip install --no-cache-dir qwen-vl-utils 'uvicorn[standard]'

# ============================================================
# Step 4: Clear compile cache (compile fresh on SageMaker)
# ============================================================
RUN rm -rf /var/tmp/neuron-compile-cache/* && \\
    mkdir -p /var/tmp/neuron-compile-cache

# Create directories and copy serving code
RUN mkdir -p /opt/ml/model /opt/ml/code /tmp/transformers_cache /tmp/hf_home
COPY code/model.py /opt/ml/code/model.py

WORKDIR /opt/ml/code

# Health check with long start period for Neuron compilation
HEALTHCHECK --interval=30s --timeout=30s --start-period=900s --retries=3 \\
    CMD curl -f http://localhost:8080/ping || exit 1

EXPOSE 8080

# SageMaker entrypoint
RUN printf '#!/bin/bash\\n\\
set -e\\n\\
echo "Starting Neuron vLLM BYOC container..."\\n\\
echo "Model files in /opt/ml/model/:"\\n\\
ls /opt/ml/model/ 2>/dev/null | head -20 || echo "(empty)"\\n\\
rm -rf /var/tmp/neuron-compile-cache/*\\n\\
echo "Compile cache cleared"\\n\\
if [ -f /opt/ml/model/model.py ]; then\\n\\
    echo "Using model.py from /opt/ml/model/"\\n\\
    cp /opt/ml/model/model.py /opt/ml/code/model.py\\n\\
fi\\n\\
echo "Starting model server..."\\n\\
exec python3 /opt/ml/code/model.py\\n\\
' > /usr/local/bin/serve && chmod +x /usr/local/bin/serve

ENTRYPOINT ["/usr/local/bin/serve"]
CMD ["serve"]
"""

with open(f"{BUILD_DIR}/Dockerfile", "w") as f:
    f.write(dockerfile)

print(f"Dockerfile written to {BUILD_DIR}/Dockerfile")
print(f"Size: {len(dockerfile)} bytes")

## Step 3: Create the Serving Code (model.py)

This FastAPI server handles:
- Model loading via vLLM with NxD Inference backend
- Compilation from scratch on the SageMaker instance
- OpenAI-compatible chat completions API at `/invocations`
- Health checks at `/ping`
- Text-only and image+text (VLM) inference

### Customization

Edit the configuration section at the top of model.py to change:
- `TP_DEGREE` -- must match your instance's NeuronCore count
- `SEQ_LEN` -- max sequence length (affects memory and compile time)
- `TEXT_NEURON_CONFIG` / `VISION_NEURON_CONFIG` -- Neuron compilation options

For a **text-only model** (e.g., Llama, Qwen), remove the `VISION_NEURON_CONFIG` and the image handling code.

In [ ]:
# Download model.py from the NeuronStuff repo
# Or write your own serving code following the same pattern
import urllib.request

MODEL_PY_URL = "https://raw.githubusercontent.com/jimburtoft/NeuronStuff/main/qwen3-vl-4b/sagemaker-byoc/code/model.py"

print("Downloading model.py...")
urllib.request.urlretrieve(MODEL_PY_URL, f"{BUILD_DIR}/code/model.py")

# Verify
size = os.path.getsize(f"{BUILD_DIR}/code/model.py")
print(f"model.py downloaded: {size} bytes")
print(f"\nBuild directory contents:")
for root, dirs, files in os.walk(BUILD_DIR):
    level = root.replace(BUILD_DIR, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = ' ' * 2 * (level + 1)
    for file in files:
        fpath = os.path.join(root, file)
        print(f"{subindent}{file} ({os.path.getsize(fpath)} bytes)")

## Step 4: Build and Push the Container

This step:
1. Creates the ECR repository (if it doesn't exist)
2. Authenticates Docker to both the DLC registry and your ECR
3. Builds the container (~10-15 minutes, mostly pip install)
4. Pushes to ECR

**Disk space**: The build requires ~20 GB. SageMaker notebook instances have sufficient space by default.

In [ ]:
%%bash -s "$ACCOUNT_ID" "$REGION" "$ECR_REPO"
ACCOUNT_ID=$1
REGION=$2
ECR_REPO=$3

# Create ECR repo if needed
aws ecr describe-repositories --repository-names $ECR_REPO --region $REGION 2>/dev/null || \
  aws ecr create-repository --repository-name $ECR_REPO --region $REGION

echo "ECR repository ready: $ECR_REPO"

In [ ]:
%%bash -s "$ACCOUNT_ID" "$REGION" "$ECR_REPO" "$ECR_TAG" "$BUILD_DIR"
set -e
ACCOUNT_ID=$1
REGION=$2
ECR_REPO=$3
ECR_TAG=$4
BUILD_DIR=$5
IMAGE_URI="${ACCOUNT_ID}.dkr.ecr.${REGION}.amazonaws.com/${ECR_REPO}:${ECR_TAG}"

echo "=== Authenticating to ECR ==="
# DLC base image registry
aws ecr get-login-password --region $REGION | \
  docker login --username AWS --password-stdin 763104351884.dkr.ecr.${REGION}.amazonaws.com

# Your ECR registry
aws ecr get-login-password --region $REGION | \
  docker login --username AWS --password-stdin ${ACCOUNT_ID}.dkr.ecr.${REGION}.amazonaws.com

echo ""
echo "=== Building container ==="
echo "This takes ~10-15 minutes (mostly pip install of vLLM + dependencies)."
echo ""

cd $BUILD_DIR
docker build -t $IMAGE_URI .

echo ""
echo "=== Pushing to ECR ==="
docker push $IMAGE_URI

echo ""
echo "Container pushed: $IMAGE_URI"

## Step 5: Verify the Container

Quick local smoke test to verify the container starts correctly.

In [ ]:
%%bash -s "$IMAGE_URI"
IMAGE_URI=$1

echo "=== Verifying container imports ==="
docker run --rm --entrypoint python3 $IMAGE_URI -c "
import torch_neuronx; print(f'torch-neuronx {torch_neuronx.__version__}')
import vllm; print(f'vLLM {vllm.__version__}')
import vllm_neuron; print('vllm_neuron plugin loaded')
import neuronx_distributed_inference; print('NxD Inference loaded')
print('All imports OK')
"

## Step 6: Deploy to SageMaker

Now use the endpoint notebook (`qwen3vl_4b_sagemaker_endpoint.ipynb`) to deploy this container.

Key parameters for the endpoint notebook:

```python
IMAGE_URI = "<your-account>.dkr.ecr.<region>.amazonaws.com/qwen3vl-neuron-byoc:v1"
INSTANCE_TYPE = "ml.inf2.8xlarge"
INFERENCE_AMI_VERSION = "al2-ami-sagemaker-inference-neuron-2"  # CRITICAL
```

The `InferenceAmiVersion` parameter is **required** -- it provides Neuron host driver v2.19, which supports the Extended ISA instructions generated by the NxD Inference compiler.

In [ ]:
print("=" * 60)
print("BUILD COMPLETE")
print("=" * 60)
print(f"")
print(f"Container: {IMAGE_URI}")
print(f"")
print(f"Next steps:")
print(f"  1. Upload model weights to S3")
print(f"  2. Open qwen3vl_4b_sagemaker_endpoint.ipynb")
print(f"  3. Set IMAGE_URI = \"{IMAGE_URI}\"")
print(f"  4. Run the deployment cells")
print(f"")
print(f"IMPORTANT: The endpoint config MUST include:")
print(f"  InferenceAmiVersion = 'al2-ami-sagemaker-inference-neuron-2'")

## Technical Notes

### Package Versions

| Package | Version | Source |
|---------|---------|--------|
| PyTorch | 2.9.0 | DLC base image |
| torch-neuronx | 2.9.0.x | DLC base image |
| neuronx-cc | 2.22.x | DLC base image |
| neuronx-distributed-inference | 0.7.x | DLC base image |
| vLLM | 0.13.0 | Installed by vllm-neuron |
| vllm-neuron | 0.4.1 | GitHub: vllm-project/vllm-neuron |

### Supported Models (vllm-neuron 0.4.1)

- Llama 2/3.1/3.3
- Qwen 2.5, Qwen 3
- Qwen2-VL, **Qwen3-VL**
- Pixtral

### Adapting for Other Models

To deploy a different model:

1. **Edit model.py** -- change `TP_DEGREE`, `SEQ_LEN`, and the neuron configs
2. **For text-only models** -- remove the `VISION_NEURON_CONFIG` and image handling
3. **For larger models** -- increase `TP_DEGREE` and use `ml.inf2.24xlarge` (6 cores) or `ml.trn1.32xlarge` (32 cores)

### Known Issues

1. **Sampler import bug**: vllm-neuron 0.4.1 has a bug where `from vllm.v1.sample import sampler as Sampler` imports the module instead of the class. The Dockerfile patches this with `sed`.

2. **tie_word_embeddings**: Qwen3-VL-4B and other models with `tie_word_embeddings=true` need a monkey-patch to copy `embed_tokens.weight` to `lm_head.weight`. This is handled in model.py.

3. **Read-only /opt/ml/model**: SageMaker mounts model data as read-only, but NxD Inference needs to write compiled artifacts under the model path. model.py creates a writable overlay at `/tmp/model` with symlinks.

### References

- [vllm-neuron GitHub](https://github.com/vllm-project/vllm-neuron) -- official plugin
- [Neuron SDK manual install](https://awsdocs-neuron.readthedocs-hosted.com/en/latest/setup/pytorch/manual.html) -- driver/tools setup
- [NxD Inference docs](https://awsdocs-neuron.readthedocs-hosted.com/en/latest/libraries/nxd-inference/index.html) -- model configuration
- [SageMaker InferenceAmiVersion](https://docs.aws.amazon.com/sagemaker/latest/APIReference/API_ProductionVariant.html) -- host AMI selection